# 03 - Segmentacion K-Means sobre RFM
Determinacion del numero optimo de clusters (metodo Elbow + Silhouette)
y entrenamiento del modelo K-Means sobre las tres dimensiones RFM.

## Importar dependencias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append("../src")
from segmentacion import calcular_elbow, entrenar_kmeans, asignar_clusters
from conexion import guardar_csv

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
os.makedirs("../outputs/graficos", exist_ok=True)

## Cargar los datos preparados

In [ ]:
df = pd.read_csv("../data/clientes_rfm_preparado.csv")
X_scaled = np.load("../data/X_rfm_scaled.npy")

print(f"DataFrame cargado: {df.shape}")
print(f"X_scaled cargado: {X_scaled.shape}")

## Metodo Elbow: numero optimo de clusters

In [ ]:
K_range, inertias, silhouette_scores = calcular_elbow(X_scaled, max_clusters=10)

print("Puntajes por numero de clusters:")
for k, inertia, sil in zip(K_range, inertias, silhouette_scores):
    print(f"k={k}: Inercia={inertia:.2f}, Silhouette={sil:.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Numero de Clusters (k)')
ax1.set_ylabel('Inercia')
ax1.set_title('Metodo Elbow - Inercia')
ax1.grid(True)

ax2.plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
ax2.set_xlabel('Numero de Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score por k')
ax2.grid(True)

plt.tight_layout()
plt.savefig('../outputs/graficos/elbow_silhouette.png', dpi=300, bbox_inches='tight')
plt.show()

## Entrenar el modelo K-Means (k=4)

In [ ]:
n_clusters = 4
kmeans, clusters = entrenar_kmeans(X_scaled, n_clusters=n_clusters)

## Asignar clusters y guardar

In [ ]:
df = asignar_clusters(df, clusters)
print("\nDistribucion de clusters:")
print(df['Cluster'].value_counts().sort_index())

guardar_csv(df, "../data/clientes_clusterizados.csv")
print("Archivo clientes_clusterizados.csv guardado")

## Visualizar la distribucion de clientes por cluster

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
cluster_counts = df['Cluster'].value_counts().sort_index()
colors = sns.color_palette('husl', n_clusters)
cluster_counts.plot(kind='bar', ax=ax, color=colors)
ax.set_xlabel('Cluster')
ax.set_ylabel('Numero de Clientes')
ax.set_title('Distribucion de Clientes por Cluster')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('../outputs/graficos/distribucion_clusters.png', dpi=300, bbox_inches='tight')
plt.show()